# 05 — Regime-Specific LSTM Training (3 variants)

This notebook trains **three pairs of regime-specific LSTMs** — one pair per variant that has a soft-probability ensemble: **O, A, B**. Each pair consists of a calm-regime LSTM and a volatile-regime LSTM trained on windows whose majority Viterbi state matches the target regime.

**Variant H is intentionally not here** — H is a single-LSTM architecture that consumes `p_volatile` as an input feature, not a regime-split ensemble. H's single baseline LSTM was trained in nb 04 and evaluated directly (no regime blending needed).

| Variant | LSTM features | HMM source | Artifacts |
|---|---|---|---|
| **O** | stationary (5) | `hmm_winner_O.joblib` + `regime_probabilities_O.parquet` | `lstm_calm_O.pt`, `lstm_volatile_O.pt` |
| **A** | stationary + sentiment (7) | `hmm_winner.joblib` + `regime_probabilities.parquet` *(default paths)* | `lstm_calm.pt`, `lstm_volatile.pt` |
| **B** | A + VIX family (10) | `hmm_winner_B.joblib` + `regime_probabilities_B.parquet` | `lstm_calm_B.pt`, `lstm_volatile_B.pt` |

Each variant uses its **own** HMM's regime labels (per Design 2 from the refactor discussion). This means variant B's regime LSTMs train on windows filtered by variant-B-informed Viterbi labels, not variant A's. The research question — "does per-regime LSTM training help over the base LSTM?" — is answered independently for each variant via within-variant DM tests in nb 07.

**Each variant runs the same Phase-3b pipeline:**

1. Load splits + variant-specific `regime_probabilities_X.parquet` + `hmm_meta_X.joblib`.
2. For each regime `calm` / `volatile`:
   - Build a `RegimeWindowDataset` filtering windows by majority Viterbi state.
   - Run Optuna TPE hyperparameter search scored on regime-filtered val MSE.
   - Retrain with best params; evaluate on the **full** (unfiltered) test set — needed for ensemble blending.
   - Save `lstm_{regime}{output_suffix}.pt` + `_scaler.joblib`.

**Key design decisions (unchanged from the original regime trainer):**
- A window straddling a regime boundary is assigned to its *majority* regime (noisy labels at transition periods).
- The feature scaler is always fit on the **full** training split, not regime-filtered, so the calm and volatile LSTMs see the same input scale.
- The val DataLoader is also regime-filtered — the calm LSTM isn't penalised for bad volatile-window predictions during Optuna's early-stopping decisions.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_regime.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
sys.path.insert(0, str(REPO_ROOT))

import config

print(f"Script: {SCRIPT}")
print(f"Python: {sys.executable}")


## Prerequisite check

Every variant's HMM must have been trained by nb 03 before this notebook runs. We verify the six required artifact files are on disk.


In [ ]:
required_artifacts = {
    "Variant O HMM winner":      config.MODELS_DIR / "hmm_winner_O.joblib",
    "Variant O HMM meta":        config.MODELS_DIR / "hmm_meta_O.joblib",
    "Variant O regime probs":    config.DATA_PROCESSED / "regime_probabilities_O.parquet",
    "Variant A HMM winner":      config.MODELS_DIR / "hmm_winner.joblib",
    "Variant A HMM meta":        config.MODELS_DIR / "hmm_meta.joblib",
    "Variant A regime probs":    config.DATA_PROCESSED / "regime_probabilities.parquet",
    "Variant B HMM winner":      config.MODELS_DIR / "hmm_winner_B.joblib",
    "Variant B HMM meta":        config.MODELS_DIR / "hmm_meta_B.joblib",
    "Variant B regime probs":    config.DATA_PROCESSED / "regime_probabilities_B.parquet",
}
missing = {k: str(v) for k, v in required_artifacts.items() if not v.exists()}
if missing:
    raise RuntimeError(
        f"Missing HMM artifacts: {missing}\n"
        "Run nb 03 (including cells 10b and 10c) to produce these before running nb 05."
    )
print("All 9 HMM artifacts found (O / A / B × {winner, meta, regime_probs}).")


## Define variant sweep

Edit `VARIANTS` to skip one for a partial rerun. Default trains all three.


In [ ]:
VARIANTS = [
    {
        "name":            "O",
        "features":        config.LSTM_VARIANT_O_FEATURES,
        "output_suffix":   "_O",
        "regime_probs":    config.DATA_PROCESSED / "regime_probabilities_O.parquet",
        "hmm_meta":        config.MODELS_DIR / "hmm_meta_O.joblib",
    },
    {
        "name":            "A",
        "features":        config.LSTM_VARIANT_A_FEATURES,
        "output_suffix":   "",
        "regime_probs":    config.DATA_PROCESSED / "regime_probabilities.parquet",
        "hmm_meta":        config.MODELS_DIR / "hmm_meta.joblib",
    },
    {
        "name":            "B",
        "features":        config.LSTM_VARIANT_B_FEATURES,
        "output_suffix":   "_B",
        "regime_probs":    config.DATA_PROCESSED / "regime_probabilities_B.parquet",
        "hmm_meta":        config.MODELS_DIR / "hmm_meta_B.joblib",
    },
]
for v in VARIANTS:
    sfx = v["output_suffix"] or "<none>"
    print(f"  {v['name']}: suffix={sfx:<6s}  feats={len(v['features'])}  rp={v['regime_probs'].name}")


## Training sweep

Trains each variant's regime pair (calm + volatile) sequentially, streaming output live. Each variant internally runs `--regime both` which executes two back-to-back Optuna studies.

Expected wall time: ~2× baseline LSTM time per variant, so ~20-40 min per variant on CPU, faster on GPU. Total: ~60-120 min for all three on CPU, ~15-30 min on GPU.


In [ ]:
variant_outputs = {}

for v in VARIANTS:
    name = v["name"]
    ARGS = [
        "--features", *v["features"],
        "--regime", "both",
        "--output-suffix", v["output_suffix"],
        "--regime-probs-path", str(v["regime_probs"]),
        "--hmm-meta-path", str(v["hmm_meta"]),
    ]
    cmd = [sys.executable, "-u", str(SCRIPT), *ARGS]

    print("=" * 80)
    print(f"[Variant {name}]  suffix={v['output_suffix'] or '<none>'}  feats={len(v['features'])}")
    print("Command:", " ".join(cmd))
    print("-" * 80)

    captured_lines = []
    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            captured_lines.append(line)
        return_code = proc.wait()
        if return_code != 0:
            print(f"\n[Variant {name}]  FAILED with return code {return_code}")
            variant_outputs[name] = {"status": "failed", "return_code": return_code, "output": "".join(captured_lines)}
            continue
    except Exception as exc:
        print(f"\n[Variant {name}]  EXCEPTION: {exc}")
        variant_outputs[name] = {"status": "exception", "error": str(exc), "output": "".join(captured_lines)}
        continue

    variant_outputs[name] = {"status": "ok", "output": "".join(captured_lines)}
    print(f"\n[Variant {name}]  ok — regime artifacts saved with suffix '{v['output_suffix']}'.")
    print()


## Parse and display results

Extract each variant's JSON results block (emitted by `train_LSTM_regime.py` under `=== Regime-Specific LSTM Results ===`) and build a comparison table across all (variant × regime) pairs.


In [ ]:
marker = "=== Regime-Specific LSTM Results ==="

results_per_variant = {}
for name, info in variant_outputs.items():
    if info.get("status") != "ok":
        print(f"  [{name}] skipped — status={info.get('status')}")
        continue
    text = info["output"]
    idx = text.find(marker)
    if idx == -1:
        print(f"  [{name}] WARNING: results marker not found.")
        continue
    json_blob = text[idx + len(marker):].strip()
    try:
        results_per_variant[name] = json.loads(json_blob)
    except json.JSONDecodeError as exc:
        print(f"  [{name}] JSON parse failed: {exc}")
        continue

# Flatten into one row per (variant, regime) pair.
rows = []
for vname, vdata in results_per_variant.items():
    for regime, r in vdata.items():
        rows.append({
            "variant":        vname,
            "regime":         regime,
            "train_windows":  r["n_train_windows"],
            "val_windows":    r["n_val_windows"],
            "best_val_mse":   r["best_val_mse_raw"],
            "test_MSE":       r["test_metrics"]["MSE"],
            "test_RMSE":      r["test_metrics"]["RMSE"],
            "test_MAE":       r["test_metrics"]["MAE"],
            "seq_len":        r["best_params"]["seq_len"],
            "hidden_size":    r["best_params"]["hidden_size"],
            "n_layers":       r["best_params"]["n_layers"],
            "lr":             f"{r['best_params']['lr']:.2e}",
        })

comparison_df = pd.DataFrame(rows).set_index(["variant", "regime"])
print("Per-(variant × regime) LSTM comparison:")
comparison_df


## Summary (fill in after execution)

Headline observations to fill in from the table above:

- **F2 (volatile-LSTM degeneracy) check:** compare `test_RMSE` and prediction variance across the three volatile LSTMs. If one variant's volatile LSTM collapses (prediction std ≈ 1e-4) while another doesn't, that's seed-dependent F2 behaviour — consistent with the structural-data-scarcity framing.
- **Training-window asymmetry:** variant B's regime LSTMs train on a smaller population than O / A's. Note in the paper.
- **HMM regime-label differences across variants:** if O's volatile label set is noticeably different from A's, O's volatile LSTM sees a different training distribution than A's — even when they use the same 5 stationary features.

DM significance testing between variants' regime LSTMs happens in nb 07. This notebook just records per-variant point estimates.


## Verify saved artifacts

Confirm all six regime checkpoints (3 variants × 2 regimes) exist on disk with their scaler counterparts.


In [ ]:
import torch, joblib

for v in VARIANTS:
    suffix = v["output_suffix"]
    print(f"[Variant {v['name']}]  suffix='{suffix}'")
    for regime in ["calm", "volatile"]:
        pt_path     = config.MODELS_DIR / f"lstm_{regime}{suffix}.pt"
        scaler_path = config.MODELS_DIR / f"lstm_{regime}{suffix}_scaler.joblib"
        if not pt_path.exists() or not scaler_path.exists():
            print(f"  [{regime}]  MISSING — {pt_path} or {scaler_path}")
            continue
        ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
        scaler = joblib.load(scaler_path)
        print(f"  [{regime}]  features={len(ckpt['features'])}  hidden={ckpt['hyperparameters']['hidden_size']}  scaler_mean_shape={scaler.mean_.shape}")
    print()

print("Verification complete.")


## Saved artifacts

| Variant | Calm checkpoint | Volatile checkpoint |
|---|---|---|
| O | `models/lstm_calm_O.pt` | `models/lstm_volatile_O.pt` |
| A | `models/lstm_calm.pt` | `models/lstm_volatile.pt` |
| B | `models/lstm_calm_B.pt` | `models/lstm_volatile_B.pt` |

Each checkpoint ships with its matching `*_scaler.joblib`. These feed into `src/ensemble.py` (three invocations, one per variant) to produce `test_predictions{_O,,_B}.parquet` — see nb 06 for per-variant ensemble evaluation + within-variant DM tests.
